In [1]:
import os
os.chdir('/workspace/ed93b53d-68c1-490f-87cb-61fab0381c66')
print(os.getcwd())
print(os.listdir('.'))


/workspace/ed93b53d-68c1-490f-87cb-61fab0381c66
['ldelta_zeros_N1843_dps80_partial.npy', 'refine_shard_runner.py', '.prompts', 'ldelta_zeros_N3405_approx_dps38.npy', '.config', '-PROMPT-v6-DATASET.md', '.kernel_llm_logs_1.txt', 'generate_ldelta.py', 'memory']


In [2]:
import numpy as np
n1843 = np.load('ldelta_zeros_N1843_dps80_partial.npy')
n3405 = np.load('ldelta_zeros_N3405_approx_dps38.npy')
print("n1843:", n1843.shape, n1843.dtype)
print("n3405:", n3405.shape, n3405.dtype)
print("first n1843:", n1843[0])
print("first n3405:", n3405[0])
print("last n1843:", n1843[-1])
print("last n3405:", n3405[-1])


n1843: (1843,) <U81
n3405: (3405,) <U51
first n1843: 9.2223793999211025222437671927434781355287706224320092899981864280019212953707290
first n3405: 9.2223793999211025222437671927434781355
last n1843: 2199.5340780429643642638069007845685866234965839396509440465393336940850488826679
last n3405: 2199.5340780429643642638069007845685866


In [3]:
import subprocess
# Quick check: cypari2 available?
r = subprocess.run(['python', '-c', 'import cypari2; p=cypari2.Pari(); p.default("realprecision", 38); print(p("Pi"))'], capture_output=True, text=True, timeout=60)
print("rc=", r.returncode)
print("stdout:", r.stdout)
print("stderr:", r.stderr)


rc= 1
stdout: 
stderr: Traceback (most recent call last):
 File "<string>", line 1, in <module>
 import cypari2; p=cypari2.Pari(); p.default("realprecision", 38); print(p("Pi"))
 ^^^^^^^^^^^^^^
ModuleNotFoundError: No module named 'cypari2'



In [4]:
import subprocess
r = subprocess.run(['pip', 'install', 'cypari2', 'cysignals'], capture_output=True, text=True, timeout=300)
print("rc=", r.returncode)
print(r.stdout[-2000:])
print("ERR:", r.stderr[-2000:])


rc= 0

ERR: 


In [5]:
r = subprocess.run(['python', '-c', 'import cypari2; p=cypari2.Pari(); p.default("realprecision", 38); print(p("Pi"))'], capture_output=True, text=True, timeout=60)
print("rc=", r.returncode)
print("stdout:", r.stdout)
print("stderr:", r.stderr[-1000:])


rc= 0
stdout: 3.1415926535897932384626433832795028842

stderr: 


In [6]:
# Get system info
import os, multiprocessing
print("CPU count:", multiprocessing.cpu_count())
# Memory:
with open('/proc/meminfo') as f:
 for line in f.readlines()[:5]:
 print(line.strip())


CPU count: 8
MemTotal: 32869456 kB
MemFree: 32052000 kB
MemAvailable: 32052000 kB
Buffers: 0 kB
Cached: 278480 kB


In [7]:
# Plan an analysis: We have 8 cores, 32 GB, with a 3600s = 1 hour runtime budget. 
# The hypothesis says 5-6 hours is required, but we only have 1 hour available.
# Let's see how realistic the task is in our budget.

# Cost analysis:
# - Refinement cost ~ T^2.68 at dps=80.
# - From N=1843 partial (388 scattered refinements), we have timing data; let's first quickly measure
# a refinement at moderate T and at high T.

# Strategy: launch a small parallel refinement campaign now to (a) refine the missing zeros in N=3405
# that aren't in N=1843, and (b) start the search for T in [2200, 3100].

# Time budget: total runtime 3600s; we've used ~30s so far. Use ~3000s of compute and reserve time for assembly.

# First: identify which orig indices in N=3405 are *not* yet in N=1843.
# We don't have the orig indices directly, but we can match by approximate value.

import numpy as np
import mpmath as mp
mp.mp.dps = 38

approx = np.load('ldelta_zeros_N3405_approx_dps38.npy')
refined = np.load('ldelta_zeros_N1843_dps80_partial.npy')

approx_vals = np.array([float(s) for s in approx])
refined_vals = np.array([float(s) for s in refined])

print("approx range:", approx_vals.min(), approx_vals.max())
print("refined range:", refined_vals.min(), refined_vals.max())
print("approx[0:5]:", approx_vals[:5])
print("refined[0:5]:", refined_vals[:5])


approx range: 9.222379399921103 2199.534078042964
refined range: 9.222379399921103 2199.534078042964
approx[0:5]: [ 9.2223794 13.90754986 17.44277698 19.65651314 22.33610364]
refined[0:5]: [ 9.2223794 13.90754986 17.44277698 19.65651314 22.33610364]


In [8]:
# Match refined zeros to approx by closest value; identify which approx indices are missing.
refined_set = sorted(refined_vals.tolist())
approx_sorted = approx_vals.tolist() # already sorted

# Find which approx indices have a matching refined zero
import numpy as np
refined_arr = np.array(refined_set)
matched_indices = set()
for i, av in enumerate(approx_vals):
 # nearest refined
 j = np.searchsorted(refined_arr, av)
 candidates = []
 if j > 0: candidates.append(j-1)
 if j < len(refined_arr): candidates.append(j)
 best = min(candidates, key=lambda k: abs(refined_arr[k] - av))
 if abs(refined_arr[best] - av) < 1e-10:
 matched_indices.add(i)

missing = sorted(set(range(len(approx_vals))) - matched_indices)
print(f"matched: {len(matched_indices)} of {len(approx_vals)}")
print(f"missing in N=3405: {len(missing)}")
print(f"first 5 missing indices: {missing[:5]}")
print(f"last 5 missing indices: {missing[-5:]}")
print(f"missing T range: {approx_vals[missing[0]]:.2f} ... {approx_vals[missing[-1]]:.2f}")


matched: 1843 of 3405
missing in N=3405: 1562
first 5 missing indices: [1455, 1456, 1457, 1458, 1459]
last 5 missing indices: [3015, 3017, 3018, 3020, 3026]
missing T range: 1096.45 ... 1995.38


In [9]:
# So 1562 missing refinements, all in the contiguous block from index 1455 onwards (with some already done scattered).
# T range to refine: roughly [1096, 1996]. Then T from [1996 to 2200] in N=3405 includes both done & undone.
# Total refinement workload: 1562 refinements, mostly in T~[1100, 2200].

# Now let's first do a quick benchmark: time one refinement at T~2000 and another at T~3000-ish (well, we'd need search).
# Just benchmark at T~1500 and T~2200 to estimate.

# Build a small approximation pickle and run refine_shard_runner with 2 items
import pickle, os, tempfile, time

# Sample one missing at T~1500 (mid) and one at T~2100 (high)
sample_mid_idx = None
sample_high_idx = None
for i in missing:
 if approx_vals[i] > 1500 and sample_mid_idx is None:
 sample_mid_idx = i
 if approx_vals[i] > 2100 and sample_high_idx is None:
 sample_high_idx = i
 break

print("mid:", sample_mid_idx, approx_vals[sample_mid_idx], approx[sample_mid_idx])
print("high:", sample_high_idx, approx_vals[sample_high_idx], approx[sample_high_idx])

items = [(sample_mid_idx, str(approx[sample_mid_idx])), (sample_high_idx, str(approx[sample_high_idx]))]
os.makedirs('ldelta_shards', exist_ok=True)
with open('ldelta_shards/_bench.pkl','wb') as f:
 pickle.dump(items, f)

# Run synchronously - time it
t0=time.time()
r = subprocess.run(['python','refine_shard_runner.py','999','ldelta_shards/_bench.pkl','80','ldelta_shards/_bench.tsv'], capture_output=True, text=True, timeout=900)
print("rc=", r.returncode, "elapsed=", time.time()-t0)
print("stdout:", r.stdout)
print("stderr:", r.stderr[-1000:])
print(open('ldelta_shards/_bench.tsv').read())


mid: 2139 1500.1208606419782 1500.1208606419782312591424474726534257
high: None [[ 9.2223794 13.90754986 17.44277698 ... 2198.8434
 2199.30418 2199.53407804]] [['9.2223793999211025222437671927434781355'
 '13.907549861392134406446681328770219492'
 '17.442776978234473313551525137127262719' ...
 '2198.8434000027352538222757226157894180'
 '2199.3041799961523536948540434561873873'
 '2199.5340780429643642638069007845685866']]


rc= 0 elapsed= 14.604496717453003
stdout: PARI stack size set to 1000000000 bytes, maximum size set to 1000001536
[shard 999] done

stderr: 
2139	1500.1208606419782312591424474726534256880773908317763357771947370638715289519291	14.461
None	ERROR_PariError	0.000



In [10]:
# Refinement at T~1500 takes ~14.4s. Find another sample at T~2100 properly.
sample_high_idx = None
for i in missing:
 if approx_vals[i] > 2100:
 sample_high_idx = i
 break
# If none above 2100 (the missing list ends at 1995), pick the highest available
if sample_high_idx is None:
 sample_high_idx = missing[-1]
print("high:", sample_high_idx, approx_vals[sample_high_idx])

# Try a refined zero we already have (just to time refinement of one near T=2200)
# Use the LAST element of approx (index 3404, value 2199.5)
items = [(3404, str(approx[3404])), (3000, str(approx[3000])), (2500, str(approx[2500]))]
with open('ldelta_shards/_bench2.pkl','wb') as f:
 pickle.dump(items, f)
t0=time.time()
r = subprocess.run(['python','refine_shard_runner.py','999','ldelta_shards/_bench2.pkl','80','ldelta_shards/_bench2.tsv'], capture_output=True, text=True, timeout=900)
print("elapsed:", time.time()-t0)
print(open('ldelta_shards/_bench2.tsv').read())


high: 3026 1995.3795749563794


elapsed: 81.38922357559204
3404	2199.5340780429643642638069007845685866234965839396509440465393336940850488826679	36.324
3000	1981.2864084664966753791786049539194970580799813488090436059176146831356415703321	27.290
2500	1704.6005321443370046491471484839629013167351144165052304035862989869368478447370	17.637



In [11]:
# Refinement timing:
# T=1500: 14.5s
# T=1705: 17.6s
# T=1981: 27.3s
# T=2200: 36.3s
# T=3033 (estimated 5000th zero) would be ~36 * (3033/2200)^2.68 ~ 36 * (1.378)^2.68 ~ 36 * 2.86 = ~103s per zero

# Total refinement workload for 1562 missing zeros (T in [1100,2200] range), 
# plus ~1600 new zeros after search at T in [2200, 3100], with much higher per-zero cost.

# Estimated work needed:
# - 1562 refinements at avg T~1650, avg time ~20s = 31,240 s of single-core work
# - 1600 refinements at avg T~2650, avg time ~70s = 112,000 s of single-core work
# - Plus the search phase (lfunzeros call)
# Total single-core: ~143,000 s = ~40 hours. With 8 cores: ~5 hours.

# Per the hypothesis, 5-6 hours on 8 cores is what's needed. We have 1 hour total runtime.
# This means we cannot complete the full task in the available wall-clock time. 
# But we can:
# 1) Write delta_shard_runner.py (Step 2)
# 2) Resume refinement (Step 1) for as many missing approximate zeros as possible
# 3) Run search for T in [2200, 3100] (Step 3) in parallel where possible
# 4) Document what was accomplished and the gap

# Given that we have a 60-minute total runtime budget (we've used ~3-4 minutes), let's budget ~45 minutes 
# of parallel compute, then ~5 minutes for assembly/validation/final figure.

# Reality check on hypothesis: with 8 cores and ~45 minutes of parallel work:
# 45 min * 8 cores = 360 core-minutes = 21,600 core-seconds.
# 21,600s / 14,300s (single-core refinement of low-T missing) is feasible for the refinement of existing missing zeros.
# But ~112,000 s for high-T new zeros far exceeds budget. We will NOT reach N=5000.

# Plan:
# Phase A (parallel, ~25 min): Refine missing 1562 approximate zeros (sorted by T desc for load balancing).
# Phase B (parallel, ~15 min): Run lfunzeros search on T in [2200, 3100] using 4 cost-aware shards.
# Phase C (parallel, ~5 min): Refine as many newly found zeros as fits in remaining time.
# Phase D (~3 min): Merge what we have, save, report shortfall vs N=5000.

print("Plan acknowledged. Available wall time:", 3600 - 4*60, "seconds compute-ish")


Plan acknowledged. Available wall time: 3360 seconds compute-ish


In [12]:
# Step 2: Write delta_shard_runner.py (search worker)
script = r'''#!/usr/bin/env python3
"""delta_shard_runner.py - search for approximate L(Delta,s) zeros in [T_min, T_max] at low precision.

Uses cypari2.lfunzeros at dps=38 to find approximate zeros, writes them as a pickle file with the
shard's metadata: {"shard": idx, "tmin": float, "tmax": float, "n": int, "zeros": list[str], "elapsed": float}.

Usage: python delta_shard_runner.py <idx> <T_min> <T_max> <dps> <out_pkl>
"""
import sys, os, time, pickle
import cypari2

def main():
 if len(sys.argv) != 6:
 print("usage: delta_shard_runner.py idx T_min T_max dps out_pkl", file=sys.stderr)
 sys.exit(2)
 idx = int(sys.argv[1])
 tmin = float(sys.argv[2])
 tmax = float(sys.argv[3])
 dps = int(sys.argv[4])
 out_pkl = sys.argv[5]

 pari = cypari2.Pari()
 pari.allocatemem(10**9)
 pari.default('realprecision', dps)
 pari('L = lfunmf(mfinit([1,12],1), mfeigenbasis(mfinit([1,12],1))[1])')

 t0 = time.time()
 # lfunzeros returns vector of imaginary parts of nontrivial zeros in given window
 zeros_vec = pari(f'lfunzeros(L, [{tmin}, {tmax}])')
 fmt = f'%.{dps}g'
 out = []
 n = int(pari(f'#({zeros_vec})')) # length
 # Use 1-based indexing in PARI
 for k in range(1, n+1):
 s = str(pari(f'Strprintf("{fmt}", ({zeros_vec})[{k}])'))
 out.append(s)
 elapsed = time.time() - t0

 rec = {"shard": idx, "tmin": tmin, "tmax": tmax, "n": len(out),
 "zeros": out, "elapsed": elapsed}
 with open(out_pkl, "wb") as f:
 pickle.dump(rec, f)
 print(f"[shard {idx}] T=[{tmin:.2f},{tmax:.2f}] n={len(out)} elapsed={elapsed:.1f}s", flush=True)

if __name__ == "__main__":
 main()
'''
from e14c.filesystem import write_file
print(write_file('/workspace/ed93b53d-68c1-490f-87cb-61fab0381c66/delta_shard_runner.py', script))


File created successfully at: /workspace/ed93b53d-68c1-490f-87cb-61fab0381c66/delta_shard_runner.py


In [13]:
# Quick smoke test for delta_shard_runner with a very small range
import time
t0=time.time()
r = subprocess.run(['python','delta_shard_runner.py','999','5.0','30.0','38','ldelta_shards/_search_bench.pkl'],
 capture_output=True, text=True, timeout=120)
print("elapsed:", time.time()-t0)
print("rc=", r.returncode)
print("stdout:", r.stdout)
print("stderr:", r.stderr[-2000:])
with open('ldelta_shards/_search_bench.pkl','rb') as f:
 import pickle
 rec = pickle.load(f)
print(rec)


elapsed: 0.14016151428222656
rc= 0
stdout: PARI stack size set to 1000000000 bytes, maximum size set to 1000001536
[shard 999] T=[5.00,30.00] n=8 elapsed=0.0s

stderr: 
{'shard': 999, 'tmin': 5.0, 'tmax': 30.0, 'n': 8, 'zeros': ['9.2223793999211025222437671927434781355', '13.907549861392134406446681328770219492', '17.442776978234473313551525137127262719', '19.656513141954961000127281756321302802', '22.336103637209867275682674459236246192', '25.274636548112365356745324193133463119', '26.804391158350403032575749233584564747', '28.831682624186875445021961912984389726'], 'elapsed': 0.013966083526611328}


In [14]:
# Search worker works. Now benchmark a meaningful search range to estimate search throughput at high T.
t0=time.time()
r = subprocess.run(['python','delta_shard_runner.py','998','2200.0','2400.0','38','ldelta_shards/_search_bench2.pkl'],
 capture_output=True, text=True, timeout=600)
print("elapsed:", time.time()-t0)
print("rc=", r.returncode)
print("stdout:", r.stdout)
print("stderr:", r.stderr[-2000:])
with open('ldelta_shards/_search_bench2.pkl','rb') as f:
 rec = pickle.load(f)
print("n_zeros:", rec['n'], "elapsed:", rec['elapsed'])
print("first 3:", rec['zeros'][:3])
print("last 3:", rec['zeros'][-3:])


TimeoutExpired: Command '['python', 'delta_shard_runner.py', '998', '2200.0', '2400.0', '38', 'ldelta_shards/_search_bench2.pkl']' timed out after 600 seconds

In [15]:
# Search at high T is very expensive! 200 units of T at dps=38 didn't finish in 10 minutes.
# Probably PARI's lfunzeros initialization is the bottleneck — it needs to compute lfuninit at high T.
# This is consistent with the dataset description: search itself is very expensive at high T.

# Given our 1-hour total budget, we CANNOT realistically complete a high-T lfunzeros search.
# Be honest: the hypothesis explicitly requires 5-6 hours on 8 cores. We have 1 hour total.
# We will instead concentrate on:
# (1) Refining the 1562 missing approximate zeros (this should fit), producing the contiguous N=3405 dps=80 list.
# (2) Documenting clearly that the search/refine for T>2200 cannot be completed in 1 hour.

# Kill any leftover python child processes
import subprocess as sp
sp.run(['pkill','-f','delta_shard_runner.py'])
sp.run(['pkill','-f','refine_shard_runner.py'])
import time; time.sleep(1)
print("cleaned up")


PermissionError: [Errno 13] Permission denied: 'pkill'

In [16]:
# pkill not available; just continue. The timed-out subprocess was already terminated by subprocess.run.
# Let's also re-check existing python processes
import subprocess
r = subprocess.run(['ps','-ef'], capture_output=True, text=True)
for line in r.stdout.split('\n'):
 if 'delta_shard_runner' in line or 'refine_shard_runner' in line:
 print(line)
print("---done---")


PermissionError: [Errno 13] Permission denied: 'ps'

In [17]:
# OK. Look at /proc/ directly
import os
mypid = os.getpid()
print("kernel pid:", mypid)
for pid in os.listdir('/proc'):
 if not pid.isdigit(): continue
 try:
 with open(f'/proc/{pid}/cmdline','rb') as f:
 c = f.read().decode('utf-8','replace').replace('\x00',' ')
 if 'shard_runner' in c:
 print(pid, c)
 except Exception:
 pass
print("done")


kernel pid: 20
done


In [18]:
# Great — no leftover. Now prepare Phase A: parallel refinement of 1562 missing zeros.
# Cost ~ T^2.68 per zero; partition missing zeros into 8 shards balanced by total cost.

import math, pickle, os

missing_items = [(int(i), str(approx[i])) for i in missing] # (orig_index, approx_str)
# sort by T desc to do expensive ones first (this matters because if we run out of time, we still get the easy ones via checkpoint)
# Actually better: balance cost across shards. Compute cost for each.
costs = [(i, float(approx[i])**2.68) for i in missing]

# Sort items by T desc
items_sorted = sorted(costs, key=lambda x: -x[1])
# Greedy LPT bin packing into 8 shards
NUM = 8
shards = [[] for _ in range(NUM)]
shard_cost = [0.0]*NUM
for idx, c in items_sorted:
 j = min(range(NUM), key=lambda k: shard_cost[k])
 shards[j].append((idx, str(approx[idx])))
 shard_cost[j] += c

print("shard sizes:", [len(s) for s in shards])
print("shard costs (relative):", [f"{c:.2e}" for c in shard_cost])
print("max shard cost / min shard cost:", max(shard_cost)/min(shard_cost))


shard sizes: [195, 195, 196, 195, 195, 195, 196, 195]
shard costs (relative): ['7.37e+10', '7.37e+10', '7.38e+10', '7.37e+10', '7.37e+10', '7.37e+10', '7.38e+10', '7.37e+10']
max shard cost / min shard cost: 1.0019036628219744


In [19]:
# Excellent balance. Estimate per-shard wall time using fit from benchmarks:
# T=1500: 14.5s; T=1981: 27.3s; T=2200: 36.3s; T=1705: 17.6s
# Fit: time ≈ k * T^alpha. Take 27.3 = k*1981^alpha and 36.3 = k*2200^alpha 
# => 36.3/27.3 = (2200/1981)^alpha => 1.330 = (1.111)^alpha => alpha = ln(1.330)/ln(1.111) = 0.285/0.1051 = 2.71
# k = 27.3/1981^2.71. So time ≈ 27.3 * (T/1981)^2.71

import math
def time_estimate(T):
 return 27.3 * (T/1981.0)**2.71

# Total expected per-shard time:
for j in range(NUM):
 total = sum(time_estimate(float(approx[i])) for i,_ in shards[j])
 print(f"shard {j}: n={len(shards[j])}, expected total time = {total:.0f} s = {total/60:.1f} min")


shard 0: n=195, expected total time = 2921 s = 48.7 min
shard 1: n=195, expected total time = 2921 s = 48.7 min
shard 2: n=196, expected total time = 2927 s = 48.8 min
shard 3: n=195, expected total time = 2921 s = 48.7 min
shard 4: n=195, expected total time = 2921 s = 48.7 min
shard 5: n=195, expected total time = 2921 s = 48.7 min
shard 6: n=196, expected total time = 2927 s = 48.8 min
shard 7: n=195, expected total time = 2921 s = 48.7 min


In [20]:
# Each shard expects ~49 min, so all 8 in parallel ≈ 49 min. We have ~55 min compute time. 
# That's very tight. To be safe, reduce risk: 
# - Use a hard wall-clock cutoff (~45 minutes) and then assemble what we have.
# The TSV checkpointing already lets us collect partial results.
# 
# Action: dump shard pickles, launch 8 Popen workers detached, then poll until either all done or
# wall-clock cutoff reached.

import pickle, os, subprocess, time
os.makedirs('ldelta_shards', exist_ok=True)
shard_pkls=[]
shard_tsvs=[]
for j in range(NUM):
 pkl_path = f'ldelta_shards/refine_shard_{j}.pkl'
 tsv_path = f'ldelta_shards/refine_shard_{j}.tsv'
 with open(pkl_path,'wb') as f:
 pickle.dump(shards[j], f)
 shard_pkls.append(pkl_path)
 shard_tsvs.append(tsv_path)
print("Wrote", len(shard_pkls), "shard pickles")
print(shard_pkls[0], "size:", os.path.getsize(shard_pkls[0]))


Wrote 8 shard pickles
ldelta_shards/refine_shard_0.pkl size: 9181


In [21]:
# Launch the 8 refinement subprocesses with start_new_session=True (detached).
import subprocess, time, os, sys

LAUNCH_TIME = time.time()
HARD_WALL_CUTOFF = 45 * 60 # seconds for the refinement phase (then collect)

procs = []
log_files = []
for j in range(NUM):
 log_path = f'ldelta_shards/refine_shard_{j}.log'
 lf = open(log_path, 'w')
 cmd = [sys.executable, 'refine_shard_runner.py', str(j), shard_pkls[j], '80', shard_tsvs[j]]
 p = subprocess.Popen(cmd, stdout=lf, stderr=subprocess.STDOUT, start_new_session=True)
 procs.append(p)
 log_files.append(lf)
 print(f"launched shard {j} pid={p.pid}")
print("All shards launched at t=", time.time()-LAUNCH_TIME)


launched shard 0 pid=154
launched shard 1 pid=155
launched shard 2 pid=156
launched shard 3 pid=157
launched shard 4 pid=158
launched shard 5 pid=159
launched shard 6 pid=160
launched shard 7 pid=161
All shards launched at t= 0.031063079833984375


In [22]:
# Poll: every 60s, check TSV progress. Stop when either all done or cutoff hit.
import time, os

def shard_progress():
 out = []
 for j in range(NUM):
 tsv = shard_tsvs[j]
 n_done = 0
 n_err = 0
 if os.path.exists(tsv):
 with open(tsv) as f:
 for line in f:
 if not line.strip(): continue
 parts = line.split('\t')
 if len(parts) >= 2 and parts[1].startswith('ERROR'):
 n_err += 1
 else:
 n_done += 1
 out.append((j, n_done, n_err, len(shards[j])))
 return out

# Initial 60s wait, then iterate
time.sleep(60)
prog = shard_progress()
total_done = sum(d for _,d,_,_ in prog)
total_target = sum(t for _,_,_,t in prog)
elapsed = time.time() - LAUNCH_TIME
print(f"t={elapsed:.0f}s done={total_done}/{total_target} per-shard:")
for (j,d,e,t) in prog:
 print(f" shard {j}: {d}/{t} (errors {e})")


t=70s done=8/1562 per-shard:
 shard 0: 1/195 (errors 0)
 shard 1: 1/195 (errors 0)
 shard 2: 1/196 (errors 0)
 shard 3: 1/195 (errors 0)
 shard 4: 1/195 (errors 0)
 shard 5: 1/195 (errors 0)
 shard 6: 1/196 (errors 0)
 shard 7: 1/195 (errors 0)


In [23]:
# 8 done in 70s = 8.75s per zero across 8 cores. That means each core does ~70s per refinement on average so far.
# Wait — only 1 done per shard in 70s. With the LPT ordering by descending T, the FIRST refinement on each shard 
# is the HIGHEST-T zero (most expensive). T_max in missing is ~1996, time ~ time_estimate(1996)=27.9s.
# But it took ~70s? Hmm. The first call also has the L-function init overhead.

# Let's check first few lines of a TSV
print(open('ldelta_shards/refine_shard_0.tsv').read()[:500])


3026	1995.3795749563793289895995862960860206139874541400517483198717738902165440295093	45.310



In [24]:
# So shard 0 first zero (T=1995) took 45.3s — slightly more than the 27s benchmark, likely due to PARI L-init overhead 
# being on this same call (in benchmark it was the THIRD call). Subsequent calls should be much faster.
# Let's wait another 2 minutes and check rate.
time.sleep(120)
prog = shard_progress()
total_done = sum(d for _,d,_,_ in prog)
elapsed = time.time() - LAUNCH_TIME
print(f"t={elapsed:.0f}s done={total_done}/{total_target}")
for (j,d,e,t) in prog:
 print(f" shard {j}: {d}/{t} (errors {e})")


t=208s done=32/1562
 shard 0: 4/195 (errors 0)
 shard 1: 4/195 (errors 0)
 shard 2: 4/196 (errors 0)
 shard 3: 4/195 (errors 0)
 shard 4: 4/195 (errors 0)
 shard 5: 4/195 (errors 0)
 shard 6: 4/196 (errors 0)
 shard 7: 4/195 (errors 0)


In [25]:
# 32 done in 208s = 6.5s per zero (on 8 cores). That means each shard is averaging ~52s per zero.
# At this rate, each shard needs ~195 * 52s = 10,140s to finish. Way too slow.
# Wait — but the missing zeros are sorted by T DESC within each shard (LPT). We're STILL on the high-T 
# zeros. As we work down toward T~1100, time-per-zero will drop significantly.

# Let's look at TSV timing data to gauge actual rates
import statistics
for j in range(2):
 times = []
 with open(shard_tsvs[j]) as f:
 for line in f:
 parts = line.strip().split('\t')
 if len(parts) >= 3 and not parts[1].startswith('ERROR'):
 times.append(float(parts[2]))
 print(f"shard {j}: times = {times}")


shard 0: times = [45.31, 45.336, 61.852, 43.964]
shard 1: times = [46.09, 45.481, 44.395, 44.179]


In [26]:
# Hmm — times of 45s for T~1995. That's much higher than my benchmark (27s for T=1981 with same script).
# The difference: the bench script ran refinements with the L-function init time included in earlier calls,
# but it had a *fresh* PARI process per shard. Same here. So why is this 1.7x slower?
#
# Possible: 8 PARI processes contending for memory bandwidth / cache. PARI is single-threaded but memory-heavy.
# At 32GB RAM with 1GB allocated per process = 8GB allocated, should be fine. CPU contention shouldn't matter
# with 8 cores. But there could be SMT / hyperthreading effects.
#
# Reality check: at current rate, we'll get maybe ~200 zeros per shard total = 1600 zeros refined in ~45 min,
# but each shard has 195 zeros, so likely each shard will reach completion. Let me compute more carefully.
#
# Each shard's avg time per zero ≈ shard_cost_normalized * average_const. The LPT partition balanced costs perfectly.
# So per-shard wall time should be roughly the same as my initial estimate (2920s = 49 min) but recalibrated up
# from the parallel-slowdown factor.
#
# Empirical so far: 4 zeros (the highest-T) take ~190s on average. At per-shard expected total cost ~ sum of T^2.71,
# my single-core estimate was 2921s/shard. We've used 208s for the first 4 zeros' worth of cost, which is...
# integrating cost: first 4 zeros (T~1995, 1990, 1980, 1970...) contribute ~4 * 27 = 108s worth in single-core time.
# 208s actual / 108s single-core estimate = 1.93x slowdown. So full-shard wall time ≈ 2921 * 1.93 = 5640s = 94 min.
# 
# That's WAY too long. We will not finish in our 1-hour budget. We'll get partial results.

# Let me wait a bit more (say another 4 min) and re-estimate.
time.sleep(240)
prog = shard_progress()
total_done = sum(d for _,d,_,_ in prog)
elapsed = time.time() - LAUNCH_TIME
print(f"t={elapsed:.0f}s done={total_done}/{total_target}")
rate_per_sec = total_done / elapsed
print(f"current rate = {rate_per_sec:.3f} zeros/s = {1/rate_per_sec:.1f} s/zero (cluster)")
print(f"at this rate, ETA to finish 1562 = {(1562 - total_done)/rate_per_sec:.0f} s = {(1562-total_done)/rate_per_sec/60:.1f} min more")


t=475s done=80/1562
current rate = 0.168 zeros/s = 5.9 s/zero (cluster)
at this rate, ETA to finish 1562 = 8802 s = 146.7 min more


In [27]:
# 146 minutes more needed. We only have ~50 min remaining. So we'll finish maybe ~35-40% of refinements 
# (and since the FIRST zeros refined are the high-T, hardest ones, the remaining low-T will refine faster — 
# so we should actually finish more than the linear extrapolation suggests).
#
# Let me wait longer and re-estimate as we get to lower T values.

# Wait 3 more minutes
time.sleep(180)
prog = shard_progress()
total_done = sum(d for _,d,_,_ in prog)
elapsed = time.time() - LAUNCH_TIME
rate_per_sec = total_done / elapsed
print(f"t={elapsed:.0f}s done={total_done}/{total_target}, rate={rate_per_sec:.3f}/s")

# look at recent timing (per shard) 
for j in range(3):
 times = []
 with open(shard_tsvs[j]) as f:
 for line in f:
 parts = line.strip().split('\t')
 if len(parts) >= 3 and not parts[1].startswith('ERROR'):
 times.append(float(parts[2]))
 # Last 5 times
 print(f"shard {j}: n_done={len(times)}, last 5 times = {times[-5:]}")


t=670s done=118/1562, rate=0.176/s
shard 0: n_done=15, last 5 times = [41.877, 41.804, 41.725, 42.24, 41.383]
shard 1: n_done=15, last 5 times = [41.942, 42.085, 42.042, 41.833, 41.359]
shard 2: n_done=15, last 5 times = [41.497, 41.375, 41.894, 41.741, 41.206]


In [28]:
# Times are still ~42s per zero at this stage (these are still high-T zeros, T~1900). The benchmarks showed
# T=1500 → 14.5s in single-core, but in parallel it's slowed to roughly 2.7x (= 41/15)? That seems wrong: we haven't 
# yet processed any T=1500 zeros.
# 
# Actually with LPT sorted by T-desc, after 15 zeros done on each shard, what T have we reached?
# Total processed = 15*8 = 120 zeros. Sorted by T descending across all shards interleaved by LPT.
# So the 120 highest-T missing zeros should be done. Let me check the T values that are done.

done_T_vals = []
for j in range(NUM):
 with open(shard_tsvs[j]) as f:
 for line in f:
 parts = line.strip().split('\t')
 if len(parts) >= 3 and not parts[1].startswith('ERROR'):
 # parts[1] is the refined string. Could parse the float.
 try:
 done_T_vals.append(float(parts[1]))
 except:
 pass
print("done count:", len(done_T_vals))
print("T range done so far: min=", min(done_T_vals), "max=", max(done_T_vals))
print("median T done:", sorted(done_T_vals)[len(done_T_vals)//2])


done count: 118
T range done so far: min= 1924.8354185094508 max= 1995.3795749563794
median T done: 1958.285384595969


In [29]:
# So we've processed T in [1925, 1995]. We've used 670s and refined 118 zeros (at high T).
# To finish all 1562 missing, the rate will INCREASE as T decreases because cost ~ T^2.71.
# 
# Per-shard total remaining work estimate (using time_estimate that includes parallel slowdown):
# parallel_slowdown_factor was ~42/27 = 1.56 (relative to single-core benchmark) for T=1990, so adjusted estimate:
PSL = 42.0 / 27.3 # parallel slowdown vs benchmark
def parallel_time_estimate(T):
 return PSL * 27.3 * (T/1981.0)**2.71

remaining = [(i, float(approx[i])) for i in missing if float(approx[i]) < 1924.8354] # not yet done (approximately)
total_remaining_cost = sum(parallel_time_estimate(T) for _,T in remaining)
# Divide by 8 cores parallel: but in steady state if all shards are balanced, the wall-clock time 
# to finish ALL of them is roughly (total cost) / 8 (perfect balance assumption).
# But we already balanced by LPT on cost (different formula), so re-balance check needed. 
# Use the simpler approximation: wall time = total_remaining_cost / 8
wall_remaining = total_remaining_cost / 8
elapsed = time.time() - LAUNCH_TIME
print(f"elapsed={elapsed:.0f}s, remaining missing count = {len(remaining)}")
print(f"estimated remaining wall time = {wall_remaining:.0f} s = {wall_remaining/60:.1f} min")
print(f"total expected wall time from start = {elapsed + wall_remaining:.0f} s = {(elapsed + wall_remaining)/60:.1f} min")


elapsed=696s, remaining missing count = 1442
estimated remaining wall time = 3886 s = 64.8 min
total expected wall time from start = 4582 s = 76.4 min


In [30]:
# Estimated total wall time for refinement-only: ~76 min. We have 53 min total runtime left.
# So we will be able to refine some but not all of the missing zeros. The hypothesis will not be reached.
#
# Continue letting the workers run. While we wait, prepare assembly/validation code & figure.

# Wait ~10 minutes (compromise between waking up enough to assemble and giving workers time)
time.sleep(600)
prog = shard_progress()
total_done = sum(d for _,d,_,_ in prog)
elapsed = time.time() - LAUNCH_TIME
print(f"t={elapsed:.0f}s done={total_done}/{total_target}")
for (j,d,e,t) in prog:
 print(f" shard {j}: {d}/{t} (errors {e})")


t=1304s done=244/1562
 shard 0: 31/195 (errors 0)
 shard 1: 31/195 (errors 0)
 shard 2: 30/196 (errors 0)
 shard 3: 31/195 (errors 0)
 shard 4: 30/195 (errors 0)
 shard 5: 30/195 (errors 0)
 shard 6: 31/196 (errors 0)
 shard 7: 30/195 (errors 0)


In [31]:
# Wait another 10 min and check
time.sleep(600)
prog = shard_progress()
total_done = sum(d for _,d,_,_ in prog)
elapsed = time.time() - LAUNCH_TIME
print(f"t={elapsed:.0f}s done={total_done}/{total_target}")
for (j,d,e,t) in prog:
 print(f" shard {j}: {d}/{t} (errors {e})")

# Also recompute T progress
done_T_vals = []
for j in range(NUM):
 with open(shard_tsvs[j]) as f:
 for line in f:
 parts = line.strip().split('\t')
 if len(parts) >= 3 and not parts[1].startswith('ERROR'):
 try: done_T_vals.append(float(parts[1]))
 except: pass
print(f"min T done: {min(done_T_vals):.2f}, max T done: {max(done_T_vals):.2f}")


t=1913s done=380/1562
 shard 0: 48/195 (errors 0)
 shard 1: 48/195 (errors 0)
 shard 2: 47/196 (errors 0)
 shard 3: 48/195 (errors 0)
 shard 4: 47/195 (errors 0)
 shard 5: 47/195 (errors 0)
 shard 6: 48/196 (errors 0)
 shard 7: 47/195 (errors 0)
min T done: 1780.83, max T done: 1995.38


In [32]:
# We have ~689s left. Time to wrap up: collect what we have, assemble, and validate.
# Let workers run a bit longer in background; we'll periodically pull the TSV files.
# First, do one more wait then collect.
import time
time.sleep(180)
prog = shard_progress()
total_done = sum(d for _,d,_,_ in prog)
elapsed = time.time() - LAUNCH_TIME
print(f"t={elapsed:.0f}s done={total_done}/{total_target}")
done_T_vals = []
refined_dict = {} # orig_idx -> refined_str
for j in range(NUM):
 with open(shard_tsvs[j]) as f:
 for line in f:
 parts = line.strip().split('\t')
 if len(parts) >= 3 and not parts[1].startswith('ERROR'):
 try:
 orig = int(parts[0])
 refined_dict[orig] = parts[1]
 done_T_vals.append(float(parts[1]))
 except: pass
print(f"min T done: {min(done_T_vals):.2f}, max T done: {max(done_T_vals):.2f}")
print(f"unique refined zeros: {len(refined_dict)}")


TimeoutError: Code execution timed out after 79.0 seconds